# Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

In [ ]:
refined_df = pd.read_csv("../data/processed/heart_cleaned.csv")

In [ ]:
categorical_columns = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]

Creating Dummies using one hot encoding for the categorical columns

In [ ]:

df_encode = pd.get_dummies(refined_df, drop_first=True, columns=categorical_columns, dtype=int)
df_encode

traain test split

In [ ]:
X = df_encode.drop("HeartDisease", axis=1)
Y = df_encode["HeartDisease"]


In [ ]:
model = LogisticRegression()
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.30, random_state=42, stratify=Y)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model.fit(X_train_scaled, Y_train)

In [ ]:
from sklearn.metrics import r2_score
y_pred = model.predict(X_test_scaled)
# r2 = r2_score(Y_test, y_pred)

n = X_test_scaled.shape[0]
p = X_test_scaled.shape[1]
# r2_adjusted = 1 - ((1-r2) * (n-1) / (n-p-1))
# r2_adjusted
# print(n,p)

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, RocCurveDisplay, confusion_matrix, ConfusionMatrixDisplay, classification_report)
training_score = model.score(X_train_scaled,Y_train) * 100

testing_score = model.score(X_test_scaled, Y_test) * 100

print(f"The training score is : {training_score :.2f}")
print(f"The testing score is : {testing_score :.2f}")

In [ ]:
cm = confusion_matrix(Y_test, y_pred)

print(cm)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["No Disease", "Disease"]
).plot(cmap="Blues")

plt.show()

In [ ]:
print(classification_report(
    Y_test,
    y_pred,
    target_names=["No Disease", "Disease"]
))

In [ ]:
from sklearn.metrics import roc_curve
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# Calculate AUC
auc = roc_auc_score(Y_test, y_prob)

print("ROC-AUC Score:", auc)

# ROC Curve
fpr, tpr, thresholds = roc_curve(Y_test, y_prob)

plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.show()

In [ ]:
print("Model accuracy:", accuracy_score(
    Y_test, model.predict(X_test_scaled)
))

print("Model score:", model.score(X_test_scaled, Y_test))

print("Confusion matrix:")
print(confusion_matrix(
    Y_test, model.predict(X_test_scaled)
))